In [1]:
import pandas as pd
import re

# Load the datasets
markdown_df = pd.read_csv('first_500_characteristics_irts_markdown.csv')
yaml_df = pd.read_csv('first_500_characteristics_irts_yaml.csv')
repo_df = pd.read_csv('first_500_characteristics_repo.csv')



In [2]:
def clean_text(text):
    """
    Cleans the input text by:
    - Removing markdown syntax
    - Removing URLs
    - Removing special characters and emojis
    - Converting to lowercase
    """
    # Remove markdown syntax (e.g., **bold**, *italic*, headers, etc.)
    text = re.sub(r'\*\*|\*|#|>', '', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove special characters and emojis
    text = re.sub(r'[^\w\s]', '', text)
    
    # Convert to lowercase
    text = text.lower()
    
    return text


In [3]:
# Clean relevant columns in characteristics_irts_markdown.csv
markdown_df['body_cleaned'] = markdown_df['body'].apply(lambda x: clean_text(str(x)))
markdown_df['IRT_raw_cleaned'] = markdown_df['IRT_raw'].apply(lambda x: clean_text(str(x)))
# Add any other relevant text fields from markdown_df that need cleaning
markdown_df['title_cleaned'] = markdown_df['title'].apply(lambda x: clean_text(str(x)))

# Clean relevant columns in characteristics_irts_yaml.csv
yaml_df['IRT_raw_cleaned'] = yaml_df['IRT_raw'].apply(lambda x: clean_text(str(x)))
# Add any other relevant text fields from yaml_df that need cleaning

# Clean relevant columns in characteristics_repo.csv
# Assuming there are relevant text fields in repo_df that require cleaning
# Here we demonstrate cleaning 'topics', though it may vary depending on your needs
repo_df['topics_cleaned'] = repo_df['topics'].apply(lambda x: clean_text(str(x)))
# Add any other relevant text fields from repo_df that need cleaning

# Display cleaned data for verification
print("Cleaned Markdown DataFrame:")
print(markdown_df[['body', 'body_cleaned']].head())

print("\nCleaned YAML DataFrame:")
print(yaml_df[['IRT_raw', 'IRT_raw_cleaned']].head())

print("\nCleaned Repo DataFrame:")
print(repo_df[['topics', 'topics_cleaned']].head())


Cleaned Markdown DataFrame:
                                                body  \
0  \n**Describe the bug**\nA clear and concise de...   
1  \n**Before Creating an issue**\n\n- Are you ru...   
2  \nWe are a small team with limited resources. ...   
3  \n## Awesome, do you have an idea? 😍\n\nIf you...   
4  \n**Describe the bug**\nA clear and concise de...   

                                        body_cleaned  
0  \ndescribe the bug\na clear and concise descri...  
1  \nbefore creating an issue\n\n are you running...  
2  \nwe are a small team with limited resources y...  
3  \n awesome do you have an idea \n\nif you have...  
4  \ndescribe the bug\na clear and concise descri...  

Cleaned YAML DataFrame:
                                             IRT_raw  \
0  name: Bug Report\ndescription: File a bug repo...   
1  name: Spammer Detection Suggestion or Problem\...   
2  name: Bug report\ndescription: Report a bug\nl...   
3  name: Feature request\ndescription: Suggest ne...   


In [4]:
print(markdown_df.columns)


Index(['name', 'about', 'title', 'labels', 'assignees', 'body', 'IRT_name',
       'full_name', 'has_initial_table', 'IRT_raw', 'IRT_full_name',
       'headlines', 'body_anonymized', 'body_cleaned', 'IRT_raw_cleaned',
       'title_cleaned'],
      dtype='object')


In [ ]:
!pip install transformers torch pandas scikit-learn

In [ ]:
!pip install --upgrade transformers


In [5]:
import pandas as pd
import torch
from transformers import RobertaTokenizer, RobertaModel
from sklearn.preprocessing import StandardScaler


# Set up the device for GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the tokenizer and model using the correct classes for CodeBERT
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
model = RobertaModel.from_pretrained("microsoft/codebert-base")
model.to(device)

def get_embeddings(text):
    """
    Generates embeddings for a given text using CodeBERT.
    """
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    # Pooling strategy: mean of token embeddings
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embeddings

# Generate embeddings for the 'body_cleaned' column in markdown_df
markdown_df['embeddings'] = markdown_df['body_cleaned'].apply(lambda x: get_embeddings(str(x)))

# Extract and normalize relevant numeric metadata features
metadata_features = ['total_issues_count', 'open_issues_count', 'closed_issues_count', 'stargazers_count', 'forks_count']

# Select and fill missing values in the repo_df
repo_df_selected = repo_df[metadata_features].fillna(0)

# Normalize the metadata features
scaler = StandardScaler()
normalized_metadata = scaler.fit_transform(repo_df_selected)

# Combine text embeddings with normalized metadata features
# Here we assume the rows in markdown_df correspond to rows in repo_df; adjust merging strategy as needed
combined_features = pd.concat([markdown_df['embeddings'].apply(pd.Series), pd.DataFrame(normalized_metadata)], axis=1)

# Display the combined features for verification
print(combined_features.head())


C:\Users\Samee\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
C:\Users\Samee\anaconda3\Lib\site-packages\torch\_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


        0         1         2         3         4         5         6    \
0 -0.076316  0.110700  0.214106  0.327175  0.191995 -0.247015  0.105294   
1 -0.055559  0.131603  0.216799  0.318998  0.244633 -0.264459  0.091785   
2 -0.093409  0.240280  0.315065  0.269358  0.141665 -0.062198 -0.033846   
3 -0.013588  0.199828  0.335744  0.336156  0.168632 -0.265297  0.052400   
4 -0.085579  0.140844  0.245513  0.249036  0.124127 -0.395911  0.147703   

        7         8         9    ...       763       764       765       766  \
0  0.209743  0.238883  0.143745  ...  0.184616  0.927621 -0.388586 -0.327716   
1  0.156076  0.172281  0.199050  ...  0.161535  0.970810 -0.320652 -0.342030   
2  0.224115  0.165692  0.292910  ...  0.172898  0.613551 -0.171950 -0.376686   
3  0.227699  0.238557  0.262554  ...  0.186835  0.837663 -0.198121 -0.391108   
4  0.187643  0.186180  0.176119  ...  0.104532  0.865559 -0.372164 -0.328944   

        767       0         1         2         3         4    
0  0

In [6]:
import numpy as np

priority_labels = np.random.choice(['high', 'medium', 'low'], size=len(combined_features))  # Example random labels

# If priority levels need to be numeric, you can use a mapping:
priority_mapping = {'high': 2, 'medium': 1, 'low': 0}
priority_encoded = pd.Series(priority_labels).map(priority_mapping)

# Add this as a target column to your DataFrame for training purposes
combined_features['priority'] = priority_encoded

# Display the DataFrame to confirm the target column
print(combined_features.head())

          0         1         2         3         4         5         6  \
0 -0.076316  0.110700  0.214106  0.327175  0.191995 -0.247015  0.105294   
1 -0.055559  0.131603  0.216799  0.318998  0.244633 -0.264459  0.091785   
2 -0.093409  0.240280  0.315065  0.269358  0.141665 -0.062198 -0.033846   
3 -0.013588  0.199828  0.335744  0.336156  0.168632 -0.265297  0.052400   
4 -0.085579  0.140844  0.245513  0.249036  0.124127 -0.395911  0.147703   

          7         8         9  ...       764       765       766       767  \
0  0.209743  0.238883  0.143745  ...  0.927621 -0.388586 -0.327716  0.317232   
1  0.156076  0.172281  0.199050  ...  0.970810 -0.320652 -0.342030  0.332432   
2  0.224115  0.165692  0.292910  ...  0.613551 -0.171950 -0.376686  0.448813   
3  0.227699  0.238557  0.262554  ...  0.837663 -0.198121 -0.391108  0.466303   
4  0.187643  0.186180  0.176119  ...  0.865559 -0.372164 -0.328944  0.302613   

          0         1         2         3         4  priority  
0 -0

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV

# Assuming combined_features is the DataFrame with all features and 'priority' is the target column

# Encode target labels if they are categorical
priority_labels = combined_features['priority']

# Check if target labels are already encoded numerically
if priority_labels.dtype == 'object' or priority_labels.dtype.name == 'category':
    # If labels are categorical, use LabelEncoder to convert them to numerical values
    le = LabelEncoder()
    priority_encoded = le.fit_transform(priority_labels)
else:
    priority_encoded = priority_labels

# Split data into training and test sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(combined_features.drop(columns=['priority']), priority_encoded, test_size=0.2, random_state=42)

print("Data prepared and split into training and test sets.")


Data prepared and split into training and test sets.


In [8]:
# Initialize the Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the training data
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=le.classes_ if 'le' in locals() else ['low', 'medium', 'high'])

print(f"Model Accuracy: {accuracy:.2f}")
print("Classification Report:")
print(report)


Model Accuracy: 0.36
Classification Report:
              precision    recall  f1-score   support

         low       0.41      0.30      0.35        30
      medium       0.30      0.52      0.38        31
        high       0.44      0.28      0.34        39

    accuracy                           0.36       100
   macro avg       0.38      0.37      0.36       100
weighted avg       0.39      0.36      0.36       100



In [9]:
# Define a parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Initialize GridSearchCV with the Random Forest model and the parameter grid
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Best parameters from GridSearchCV
best_params = grid_search.best_params_

# Train the model again using the best parameters
best_model = grid_search.best_estimator_

# Predict on the test set with the best model
y_pred_best = best_model.predict(X_test)

# Evaluate the best model
accuracy_best = accuracy_score(y_test, y_pred_best)
report_best = classification_report(y_test, y_pred_best, target_names=le.classes_ if 'le' in locals() else ['low', 'medium', 'high'])

print(f"Best Model Accuracy: {accuracy_best:.2f}")
print("Best Model Classification Report:")
print(report_best)
print(f"Best Parameters: {best_params}")


Fitting 3 folds for each of 216 candidates, totalling 648 fits
Best Model Accuracy: 0.38
Best Model Classification Report:
              precision    recall  f1-score   support

         low       0.40      0.27      0.32        30
      medium       0.35      0.61      0.45        31
        high       0.42      0.28      0.34        39

    accuracy                           0.38       100
   macro avg       0.39      0.39      0.37       100
weighted avg       0.39      0.38      0.37       100

Best Parameters: {'bootstrap': True, 'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 200}


In [13]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_recall_curve, roc_curve



# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# ROC-AUC Score (for binary or multiclass classification)
try:
    roc_auc = roc_auc_score(y_test, model.predict_proba(X_test), multi_class='ovr')
    print(f"ROC-AUC Score: {roc_auc:.2f}")
except:
    print("ROC-AUC not applicable for this problem type.")

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, model.predict_proba(X_test)[:, 1])
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, marker='.')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.show()


AttributeError: 'LabelEncoder' object has no attribute 'classes_'

<Figure size 800x600 with 0 Axes>

Data prepared and split into training and test sets.


NameError: name 'le' is not defined